# Portrait Animator on Colab

Runs the whole Flask app + LivePortrait on a Colab T4 GPU and exposes it via a public Cloudflare tunnel.

**Before running:** set **Runtime → Change runtime type → T4 GPU**.

You have two options to get the project into Colab:
- **A. GitHub:** push your local `ArtMLfinal` to a GitHub repo and set `REPO_URL` below.
- **B. Upload:** zip the `app/` directory locally, upload it with the file sidebar, and set `USE_UPLOADED_ZIP=True`.

Then run every cell top-to-bottom.

In [ ]:
REPO_URL = "https://github.com/YOUR_USERNAME/ArtMLfinal.git"  # <-- edit me
USE_UPLOADED_ZIP = False   # set True if you uploaded app.zip instead

In [ ]:
# 1. Verify you actually got a GPU.
!nvidia-smi | head -20

In [ ]:
# 2. Pull in the project.
import os, subprocess, shutil
os.chdir('/content')

if USE_UPLOADED_ZIP:
    # Expects /content/app.zip (drag it into the Colab file sidebar).
    assert os.path.exists('/content/app.zip'), 'Upload app.zip first.'
    shutil.rmtree('/content/ArtMLfinal', ignore_errors=True)
    os.makedirs('/content/ArtMLfinal', exist_ok=True)
    !cd /content/ArtMLfinal && unzip -q /content/app.zip
else:
    shutil.rmtree('/content/ArtMLfinal', ignore_errors=True)
    !git clone $REPO_URL /content/ArtMLfinal

os.chdir('/content/ArtMLfinal/app')
print('Working in:', os.getcwd())

In [ ]:
# 3. Clone LivePortrait (setup.sh skips if already present).
!if [ ! -d liveportrait ]; then git clone https://github.com/KwaiVGI/LivePortrait.git liveportrait; fi

In [ ]:
# 4. Install deps. Colab already has CUDA torch preinstalled — pip will skip it.
#    We install the project's requirements plus LivePortrait's Linux/CUDA requirements.
!pip install --quiet --prefer-binary -r requirements.txt
!cd liveportrait && pip install --quiet --prefer-binary -r requirements.txt

In [ ]:
# 5. Download pretrained weights from HuggingFace (~2 GB, takes 1-3 min).
!pip install --quiet --upgrade 'huggingface_hub[cli]'
!mkdir -p liveportrait/pretrained_weights
!hf download KwaiVGI/LivePortrait --local-dir liveportrait/pretrained_weights

In [ ]:
# 6. Install cloudflared (creates a public HTTPS URL to our Flask server).
!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared
!chmod +x /usr/local/bin/cloudflared
!cloudflared --version

In [ ]:
# 7. Start Flask in the background, then start the Cloudflare tunnel.
#    Stop both with the "Stop" button next to this cell (interrupt).
import subprocess, time, re, sys, os, signal

os.environ['PORT'] = '5001'

# Kill any previous runs from earlier executions of this cell.
subprocess.run(['pkill', '-f', 'app.py'], check=False)
subprocess.run(['pkill', '-f', 'cloudflared'], check=False)
time.sleep(1)

# Launch Flask (stdout/stderr -> /content/flask.log).
flask_log = open('/content/flask.log', 'w')
flask = subprocess.Popen(
    ['python', 'app.py'],
    stdout=flask_log, stderr=subprocess.STDOUT,
    cwd='/content/ArtMLfinal/app',
)
print('Flask PID:', flask.pid, '(logs: /content/flask.log)')

# Wait for Flask to bind.
for _ in range(30):
    time.sleep(1)
    r = subprocess.run(['curl', '-s', '-o', '/dev/null', '-w', '%{http_code}',
                        'http://127.0.0.1:5001/health'], capture_output=True, text=True)
    if r.stdout.strip() == '200':
        print('Flask up.')
        break
else:
    print('Flask did not come up — check /content/flask.log')
    raise SystemExit

# Launch cloudflared and capture its public URL.
tunnel = subprocess.Popen(
    ['cloudflared', 'tunnel', '--url', 'http://localhost:5001', '--no-autoupdate'],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1,
)
print('Waiting for tunnel URL...')
public_url = None
for line in tunnel.stdout:
    sys.stdout.write(line)
    m = re.search(r'https://[-\w]+\.trycloudflare\.com', line)
    if m and not public_url:
        public_url = m.group(0)
        print('\n' + '=' * 60)
        print('OPEN THIS URL IN YOUR BROWSER:')
        print(public_url)
        print('=' * 60 + '\n')
# (If the tunnel process exits, the loop ends and the cell returns.)

## Notes

- When the above cell prints an `https://…trycloudflare.com` URL, open it in any browser — works from your laptop, phone, anywhere.
- On a T4, expect ~10-30 seconds per 3-second clip.
- To stop, press the Stop button next to the running cell. To fully free the GPU, **Runtime → Disconnect and delete runtime**.
- Free Colab will disconnect after idle. Anything in `/content/` is lost on disconnect — download any generated GIFs first, or mount Google Drive.